This file is part of the CRISPRsummerschool 2026 exercises

Copyright (c) 2023-26 Christian Anthon & 2026 Gül Sude Demircan

This program is free software: you can redistribute it and/or modify
it under the terms of the GNU General Public License as published by
the Free Software Foundation, version 3.

# Warming up exercise
Below you will find the first exercise, in which you will be introduced a small CRISPR on-target model in PyTorch and use it to train a small ontarget efficiency model on real data.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RTH-tools/CRISPRsummerschool/blob/main/2026/CRISPR/exercise/crispr_2026_crispr_exercise1.ipynb)


## basic code definitions
Enter the cell below and press play or Ctrl+Enter in the block below to execute. You should see the message "Definitions executed" printed after execution.

In [ ]:
#!/usr/bin/env python3
# CRISPRsummerschool 2026
import os
import copy
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

torch.manual_seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


eLENGTH30 = 30   # sequence length: the target region is a 30-mer, so 30 input positions
eDEPTH    = 4    # one-hot channels per position: the 4 nucleotides A, C, G, T

# Function to convert DNA sequence to an index encoding (A,C,G,T/U -> 0,1,2,3)
def onehot(x):
    z = list()
    for y in list(x):
        if y in "Aa":
            z.append(0)
        elif y in "Cc":
            z.append(1)
        elif y in "Gg":
            z.append(2)
        elif y in "TtUu":
            z.append(3)
        else:
            print("Non-ATGCU character in", x)
            raise Exception
    return z

# Function to set the data into the appropriate format
def set_data(DX, s):
    if s is None:
        return
    for j, x in enumerate(onehot(s)):
        DX[j][x] = 1

# Preprocessing function for the sequence data
def preprocess_seq(data):
    DATA_X30 = np.zeros((len(data), eLENGTH30, eDEPTH), dtype=np.float32)  # onehot
    DATA_G = np.zeros((len(data), 1), dtype=np.float32)  # deltaGb
    DATA_Y = np.zeros((len(data)), dtype=np.float32)  # efficiency

    for l, d in enumerate(data):
        set_data(DATA_X30[l], d[1])
        DATA_G[l] = -d[2]
        DATA_Y[l] = d[3]
    return (DATA_X30, DATA_G, DATA_Y)


# Convert numpy arrays to torch tensors on `device`.
# NOTE: the one-hot stays (N, 30, 4) here; the model permutes it to
#       (N, 4, 30) internally, because PyTorch Conv1d expects the layout
#       (batch, channels, length).
def to_tensors(x30, g, y):
    return (torch.from_numpy(x30).to(device),
            torch.from_numpy(g).to(device),
            torch.from_numpy(y).to(device))

def evaluate(model, data):
    """Return (mse, mae) on a (Xc, Xg, y) split. Runs in eval mode (dropout OFF)."""
    Xc, Xg, y = data
    model.eval()
    with torch.no_grad():
        pred = model(Xc, Xg).squeeze(-1)          # (N, 1) -> (N,)
        mse = torch.mean((pred - y) ** 2).item()
        mae = torch.mean(torch.abs(pred - y)).item()
    return mse, mae

def train(model, train_data, val_data, epochs=200, batch_size=64, lr=1e-3,
          patience=25, min_delta=0.1, verbose=True):
    """Mini-batch training with early stopping and restore-best-weights."""
    Xc, Xg, y = train_data
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    best_val, best_state, wait, history = float("inf"), None, 0, []
    n = Xc.shape[0]
    for epoch in range(epochs):
        model.train()                             # dropout ON
        perm = torch.randperm(n, device=Xc.device)
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            optimizer.zero_grad()
            pred = model(Xc[idx], Xg[idx]).squeeze(-1)   # (B,1) -> (B,): MUST squeeze
            loss = loss_fn(pred, y[idx])
            loss.backward()
            optimizer.step()
        val_mse, val_mae = evaluate(model, val_data)
        history.append(val_mse)
        if val_mse < best_val - min_delta:        # "minimum improvement" rule
            best_val, best_state, wait = val_mse, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
        if verbose:
            print("epoch %3d  val_mse=%8.3f  val_mae=%6.3f  best=%8.3f  wait=%d"
                  % (epoch, val_mse, val_mae, best_val, wait))
        if wait >= patience:
            print("Early stopping at epoch %d (best val_mse=%.3f)" % (epoch, best_val))
            break
    if best_state is not None:
        model.load_state_dict(best_state)         # restore_best_weights=True
    return history


# ---- Robust data loader: identical behaviour in Colab and local Jupyter ----
# A file already in this folder is used as-is; a missing file is downloaded and
# its contents are validated. Pure Python (no shell), so it behaves the same in
# Colab, local Jupyter, Windows/Mac/Linux.
import urllib.request

DATA_SOURCES = {
    "training_data.csv": [
        "https://rth.dk/internal/index.php/s/S4jQMaER6nYAJGe/download",
    ],
    "validation_data.csv": [
        "https://rth.dk/internal/index.php/s/oHspJCgniRMog6r/download",
    ],
}

def _is_valid_csv(path):
    """A real data file starts with the known header, not an HTML error page."""
    try:
        with open(path, "r", encoding="utf-8", errors="ignore") as fh:
            first = fh.readline()
        return ("target" in first) and ("deltaGb" in first)
    except OSError:
        return False

def fetch_data(fname, dest_dir="."):
    """Return the path to a valid `fname`, downloading it only if needed.
    urllib raises on HTTP errors (unlike a bare `curl -o`) and we re-check the
    content, so a bad/expired URL fails loudly instead of silently writing an
    HTML page into a .csv."""
    dest = os.path.join(dest_dir, fname)
    if _is_valid_csv(dest):
        print("using existing", dest)
        return dest
    problems = []
    for url in DATA_SOURCES[fname]:
        try:
            print("downloading %s from %s ..." % (fname, url.split("/")[2]))
            urllib.request.urlretrieve(url, dest)
        except Exception as e:
            problems.append("%s -> %s" % (url, e))
            continue
        if _is_valid_csv(dest):
            return dest
        problems.append("%s -> downloaded file is not a valid CSV (an error page?)" % url)
    if os.path.exists(dest):
        os.remove(dest)                       # never leave a corrupt .csv behind
    raise RuntimeError(
        "Could not obtain %s. Upload it into this folder manually, or fix the "
        "URLs in DATA_SOURCES above.\nTried:\n  %s" % (fname, "\n  ".join(problems)))

fetch_data("training_data.csv")
fetch_data("validation_data.csv")

print('\n\nDefinitions executed')


## Exercise 1.1
The sequence of the ontarget of an example gRNA (ACTGAAAAAACCCCCTTTTT), needs to be onehot encoded. An example ontarget of ACTGAAAAAACCCCCTTTTT is TTTTACTGAAAAAACCCCCTTTTTGGGAAA, which includes a four nucleotide prefix, the ontarget to the gRNA, the PAM sequnce and a three nucleotide suffix.

**Task 1.** Split the 30-mer `TTTTACTGAAAAAACCCCCTTTTTGGGAAA` into its four parts. For each part, give the subsequence and its 0-based slice range (`s[0:4]` style): prefix (4 nt), spacer / on-target (20 nt), PAM (3 nt), suffix (3 nt).

**Task 2.** Call `onehot("ACTGAAAAAACCCCCTTTTT")` (the function defined in the cell above) and print the result. State how many numbers it returns and which values they can take.

**Task 3.** A true one-hot encoding gives every base its own length-4 vector - `A = [1,0,0,0]`, `C = [0,1,0,0]`, `G = [0,0,1,0]`, `T = [0,0,0,1]` - so a 20-nt spacer becomes a `(20, 4)` array of 0s and 1s. Compare that with your Task 2 output, then answer:

1. Is the Task 2 output a one-hot encoding? If not, what kind of encoding is it?
2. Name one concrete way this encoding could mislead a model that feeds those numbers straight into a network.

In [ ]:
#answer


## Exercise 1.2
Excecute the code below to load the data into the notebook.

In [ ]:
# x30 - onehot encoded 30mer
# g   - deltaGb
# y   - the efficiency value (~0 - ~100)

PATH = './'
d = pd.read_csv(PATH + 'training_data.csv').values.tolist()
(x30, g, y) = preprocess_seq(d)

# Validation data
dv = pd.read_csv(PATH + 'validation_data.csv').values.tolist()
(x30v, gv, yv) = preprocess_seq(dv)

# to torch tensors on `device`
x30_t, g_t, y_t = to_tensors(x30, g, y)
x30v_t, gv_t, yv_t = to_tensors(x30v, gv, yv)
print("train tensors:\n", x30_t.shape, "\n", g_t.shape, "\n", y_t.shape)

### Reading the printed shapes above

```
train tensors: 
torch.Size([15935, 30, 4]) 
torch.Size([15935, 1]) 
torch.Size([15935])
```

All three are `float32` on `device`. **15935 = number of training guides**, and it is the first axis of all three: index `i` means the same guide in each.

| tensor | shape | axes |
|---|---|---|
| `x30_t` | `(15935, 30, 4)` | guide, position in the 30-mer, nucleotide channel `A,C,G,T` |
| `g_t` | `(15935, 1)` | guide, the energy feature (`-deltaGb`) |
| `y_t` | `(15935,)` | guide (measured efficiency) |

- `x30_t`: each of the 30 position-vectors holds exactly one `1`, so every guide's matrix sums to 30. `forward` then calls `x.permute(0, 2, 1)` to get `(batch, 4, 30)`, because `Conv1d` expects `(batch, channels, length)`.
- `g_t` is `(N, 1)`, not `(N,)`, so `torch.cat([g, z], dim=1)` can glue it onto the dense activations - both operands need the same number of axes.
- `y_t` is `(N,)`, which is why training does `model(...).squeeze(-1)`: the net outputs `(B, 1)`, and without the squeeze a `(B,1)` prediction would broadcast against a `(B,)` target into a `(B,B)` loss.

Validation (`x30v_t, gv_t, yv_t`) is identical with 3984 guides.

### Exercise 1.2.1

Print the first row of the raw training data (`d[0]`) and the first entry of each processed array (`x30[0]`, `g[0]`, `y[0]`).

1. Which field of `d[0]` did each of `x30[0]`, `g[0]`, `y[0]` come from? Do the values match?
2. Is `x30[0]` what a one-hot encoding should look like?

In [ ]:
#answer

### Exercise 1.2.2 Model definition
In the code below, a simplified version of the CRISPRon ontarget model is defined. Review the code without diving into the details. Then execute it to load the model.

In [ ]:
DROPOUT_DENSE = 0.3   # dropout rate: fraction of units switched off after each dense layer (training only)
CONV_1_SIZE   = 3     # convolution kernel width, in nucleotides (a 3-base window)
N_CONV_1      = 40    # number of convolution filters (motif detectors)
N_DENSE       = 40    # width of the first dense layers (collect, dense1)
N_OUT         = 40    # width of the last hidden layers (dense2, dense3), before the 1-unit output

class SimpleCRISPRon(nn.Module):
    """A simplified version of the CRISPRon on-target model (PyTorch).

    One 1D convolution over the one-hot sequence, followed by fully connected
    (dense) layers. The binding energy dGb is concatenated in *after* the first
    dense layer.
    """
    def __init__(self, n_conv=N_CONV_1, kernel=CONV_1_SIZE, n_dense=N_DENSE,
                 n_out=N_OUT, dropout=DROPOUT_DENSE, seq_len=eLENGTH30, depth=eDEPTH):
        super().__init__()
        self.conv = nn.Conv1d(depth, n_conv, kernel)      # (B,4,30) -> (B,n_conv,28)
        conv_out_len = seq_len - kernel + 1               # 30 - 3 + 1 = 28
        flat = n_conv * conv_out_len                      # flattened conv features
        self.collect = nn.Linear(flat, n_dense)           # "dense_0"
        self.dense1 = nn.Linear(n_dense + 1, n_dense)     # "dense_1"  (+1 = raw dGb)
        self.dense2 = nn.Linear(n_dense, n_out)           # "dense_2"
        self.dense3 = nn.Linear(n_out, n_out)             # "dense_on_off"
        self.out = nn.Linear(n_out, 1)                    # output (linear, no activation)
        self.drop = nn.Dropout(dropout)
        self.apply(self._xavier_uniform_init)

    def features(self, x, g):
        # Conv1d wants (batch, channels, length); our one-hot is
        # (batch, length=30, channels=4), so swap the last two axes.
        x = x.permute(0, 2, 1)                            # (B,30,4) -> (B,4,30)
        z = torch.relu(self.conv(x))                      # convolution + ReLU
        z = torch.flatten(z, start_dim=1)                 # (B, n_conv*28)
        z = self.drop(torch.relu(self.collect(z)))        # dense_0 + ReLU + dropout
        z = torch.cat([g, z], dim=1)                      # concat raw dGb
        z = self.drop(torch.relu(self.dense1(z)))         # dense_1
        z = self.drop(torch.relu(self.dense2(z)))         # dense_2
        z = self.drop(torch.relu(self.dense3(z)))         # dense_on_off
        return z

    def forward(self, x, g):
        return self.out(self.features(x, g))              # (B, 1)

    @staticmethod
    def _xavier_uniform_init(m):
        if isinstance(m, (nn.Conv1d, nn.Linear)):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None:
                nn.init.zeros_(m.bias)

model = SimpleCRISPRon().to(device)
print(model)
print('Model defined')


### Exercise 1.2.3

`print(model)` lists the layers, but not the order they run in. For that, read the `features` method in the cell above.

1. The model takes two inputs. Name them, and say which layer each one enters.
2. Which layer produces the prediction, and what is its output shape?
3. Sort the layers into the convolutional part and the fully connected (MLP) part.
4. Where does ΔGb join the sequence path, and what effect does that have on the layer receiving it?

Useful:

```python
for name, p in model.named_parameters():
    print(name, tuple(p.shape))     # weight shapes, layer by layer
```

In [ ]:
#answer

### Exercise 1.2.4 Model training
Below you will find code for training the simplified CRISPRon model on the provided training data, using the validation data for model evaluation during training. Familiarize yourself with the code and parameters.

What is the difference between BATCH_SIZE and epochs?

Execute the model **training**

In [ ]:
print("training...")

LEARN = 1e-3       # learning rate for Adam
EPOCHS = 200       # maximum number of epochs
BATCH_SIZE = 64    # batch size for the training

# (to restart training from scratch, re-run the model-definition cell in 1.2.2)
history = train(model, (x30_t, g_t, y_t), (x30v_t, gv_t, yv_t),
                epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LEARN,
                patience=25, min_delta=0.1)

print("done")
val_mse, val_mae = evaluate(model, (x30v_t, gv_t, yv_t))
print("validation  mse=%.3f  mae=%.3f" % (val_mse, val_mae))


### Exercise 1.2.4.1
How many epochs did the code use before it stopped?

When did the training reach the optimimal model?

Does the code output the exact same performance if you run it twice? Why / Why not?

In [ ]:
#answer

### Exercise 1.2.4.2
Repeat the model initialization in (1.2.2) and the model training (1.2.4) 3-5 times and record the performance on the validation data each time.

For the best model you obtain, compare the mean squared error and mean absolute error on the validation data with the errors obtained for the **full model** trained on the same data (**mae 9.1, mse 141.3** for the **full model** which is based on the same principles, but contains more and larger layers)?

In [ ]:
# answer

### Exercise 1.2.5 (if time allows)
Play with the model and parameters to get a better performance.

Can you beat the full the model performance?

What would be the proper way to **test** that?

In [ ]:
#answer


## Reference

These exercises use a **simplified** version of the CRISPRon on-target efficiency model:

- Xiang, X., Corsi, G. I., Anthon, C., *et al.* (2021). Enhancing CRISPR-Cas9 gRNA efficiency prediction by data integration and deep learning. *Nature Communications*, **12**, 3238. https://doi.org/10.1038/s41467-021-23576-0
